# 02. Download the history
Hourly prices from REN (2019 to June 2026) and 15-minute production by source from REN (2019 to today).

In [1]:
# Import the libraries
import time
import requests
import pandas as pd

In [2]:
# Download one day of REN prices: one row per hour, one column per country
def read_ren_prices_day(day):
    url = "https://servicebus.ren.pt/datahubapi/electricity/ElectricityMarketPricesDaily"
    response = requests.get(url, params={"culture": "en-US", "date": day})
    if response.status_code != 200:
        return pd.DataFrame()
    data = response.json()
    table = pd.DataFrame({"hour": data["xAxis"]["categories"]})
    for serie in data["series"]:
        table["price_" + serie["name"].lower()] = serie["data"]
    table["date"] = day
    return table

# Download one day of REN production: one row per 15 minutes, one column per source
def read_ren_production_day(day):
    url = "https://servicebus.ren.pt/datahubapi/electricity/ElectricityProductionBreakdownDaily"
    response = requests.get(url, params={"culture": "en-US", "date": day})
    if response.status_code != 200:
        return pd.DataFrame()
    data = response.json()
    table = pd.DataFrame({"time": data["xAxis"]["categories"]})
    for serie in data["series"]:
        table[serie["name"]] = serie["data"]
    table["date"] = day
    return table

In [3]:
# Test both functions on one week before downloading everything
test_prices = []
test_production = []
for day in pd.date_range("2025-03-24", "2025-03-31"):
    day_text = day.strftime("%Y-%m-%d")
    test_prices.append(read_ren_prices_day(day_text))
    test_production.append(read_ren_production_day(day_text))

test_prices = pd.concat(test_prices)
test_production = pd.concat(test_production)
print(len(test_prices), "price rows (8 days x 24 hours = 192, minus 1 on the clock change day)")
print(len(test_production), "production rows (8 days x 96 = 768, minus 4 on the clock change day)")
test_production.head()

191 price rows (8 days x 24 hours = 192, minus 1 on the clock change day)
764 production rows (8 days x 96 = 768, minus 4 on the clock change day)


,time,Consumption + Storage,Consumption,Pumping,Hydro,Solar,Import,Export,Wind,Natural Gas,Other Thermal,Biomass,Coal,Wave,Consumption of Batteries,Battery Injection,date
0,00:00,5736.3,5632.6,103.7,3730.4,0.1,0.0,935.3,2296.5,310.3,8.1,341.9,0.0,0.0,0.0,0.0,2025-03-24
1,00:15,5718.2,5585.3,132.9,3493.7,0.1,0.0,749.2,2247.9,391.2,8.1,342.5,0.0,0.0,0.0,0.0,2025-03-24
2,00:30,5676.5,5462.9,213.6,3512.8,0.1,0.0,753.7,2185.4,396.3,8.0,345.1,0.0,0.0,0.0,0.0,2025-03-24
3,00:45,5632.1,5331.9,300.2,3459.3,0.1,0.0,659.8,2114.2,381.3,8.5,345.6,0.0,0.0,0.0,0.0,2025-03-24
4,01:00,6127.9,5230.1,897.8,2926.2,0.1,372.9,0.0,2087.1,401.5,8.4,348.4,0.0,0.0,0.0,0.0,2025-03-24


In [4]:
# Download all days: prices until 29 June 2026 (last day available on REN), production until yesterday
all_prices = []
all_production = []
failed_days = []

for day in pd.date_range("2019-01-01", "2026-10-01"):
    day_text = day.strftime("%Y-%m-%d")

    # Prices (only until the last day REN has)
    if day_text <= "2026-06-29":
        prices = read_ren_prices_day(day_text)
        if len(prices) == 0:
            failed_days.append(("prices", day_text))
        all_prices.append(prices)

    # Production
    production = read_ren_production_day(day_text)
    if len(production) == 0:
        failed_days.append(("production", day_text))
    all_production.append(production)

    # Show progress at the start of each month and wait a little between days
    if day.day == 1:
        print("Done:", day_text)
    time.sleep(0.1)


Done: 2019-01-01
Done: 2019-02-01
Done: 2019-03-01
Done: 2019-04-01
Done: 2019-05-01
Done: 2019-06-01
Done: 2019-07-01
Done: 2019-08-01
Done: 2019-09-01
Done: 2019-10-01
Done: 2019-11-01
Done: 2019-12-01
Done: 2020-01-01
Done: 2020-02-01
Done: 2020-03-01
Done: 2020-04-01
Done: 2020-05-01
Done: 2020-06-01
Done: 2020-07-01
Done: 2020-08-01
Done: 2020-09-01
Done: 2020-10-01
Done: 2020-11-01
Done: 2020-12-01
Done: 2021-01-01
Done: 2021-02-01
Done: 2021-03-01
Done: 2021-04-01
Done: 2021-05-01
Done: 2021-06-01
Done: 2021-07-01
Done: 2021-08-01
Done: 2021-09-01
Done: 2021-10-01
Done: 2021-11-01
Done: 2021-12-01
Done: 2022-01-01
Done: 2022-02-01
Done: 2022-03-01
Done: 2022-04-01
Done: 2022-05-01
Done: 2022-06-01
Done: 2022-07-01
Done: 2022-08-01
Done: 2022-09-01
Done: 2022-10-01
Done: 2022-11-01
Done: 2022-12-01
Done: 2023-01-01
Done: 2023-02-01
Done: 2023-03-01
Done: 2023-04-01
Done: 2023-05-01
Done: 2023-06-01
Done: 2023-07-01
Done: 2023-08-01
Done: 2023-09-01
Done: 2023-10-01
Done: 2023-11-

In [5]:
# Join all days and save the raw data
prices = pd.concat(all_prices)
production = pd.concat(all_production)
prices.to_csv("../data/raw/ren_prices_hourly.csv", index=False)
production.to_csv("../data/raw/ren_production_15min.csv", index=False)

print(len(prices), "price rows,", len(production), "production rows")
print(len(failed_days), "failed requests:", failed_days[:10])

65687 price rows, 271676 production rows
1 failed requests: [('production', '2022-04-29')]


In [6]:
# Try the failed day again and add it to the production table
retry = read_ren_production_day("2022-04-29")
print(len(retry), "rows downloaded on the second try")

if len(retry) > 0:
    production = pd.concat([production, retry])
    production.to_csv("../data/raw/ren_production_15min.csv", index=False)
    print("Saved:", len(production), "production rows")

96 rows downloaded on the second try
Saved: 271772 production rows


In [7]:
# Look at how the price hours are written
print(prices["hour"].unique()[:30])
prices.head()

<StringArray>
[ '1',  '2',  '3',  '4',  '5',  '6',  '7',  '8',  '9', '10', '11', '12', '13',
 '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25']
Length: 25, dtype: str


,hour,price_pt,price_es,date
0,1,66.88,66.88,2019-01-01
1,2,66.88,66.88,2019-01-01
2,3,66.00,66.00,2019-01-01
3,4,63.64,63.64,2019-01-01
4,5,58.85,58.85,2019-01-01
